# Thalamus — fish species classifier (Colab)

Colab-friendly wrapper around `scripts/fish_classifier_train.py`. The training
logic lives in the repo, not in this notebook, so a run here and a run on a
workstation stay identical.

**Runtime → Change runtime type → GPU** before you start.

Steps: install deps → get data → set config → train → export → download.

## 1. Check the GPU

If this reports no GPU, training will still run but will be roughly an order of
magnitude slower.

In [ ]:
!nvidia-smi || echo 'No GPU allocated — check Runtime > Change runtime type'

## 2b. (Optional) Build the Florida fish dataset from iNaturalist

Run this section only if you need to (re)generate the raw dataset from the
[iNaturalist Open Dataset](https://github.com/inaturalist/inaturalist-open-data).
It filters the metadata down to ray-finned fish (Actinopterygii) observed in
Florida waters, then downloads the matching research-grade photos into
`data/raw_florida_fish/`, organized one folder per species.

This is a large download and can take a while. If you already have the dataset
in Drive (Step 3), skip this.

In [ ]:
# Install deps and pull the iNaturalist metadata (~ several GB of gzipped CSVs).
!pip install -q duckdb pandas boto3 awscli

import os

INAT_DATA_DIR = '/content/inat_data'
os.makedirs(INAT_DATA_DIR, exist_ok=True)

for name in ['taxa', 'observations', 'photos']:
    dest = f'{INAT_DATA_DIR}/{name}.csv.gz'
    if not os.path.exists(dest):
        !aws s3 cp s3://inaturalist-open-data/{name}.csv.gz {dest} --no-sign-request
    else:
        print(f'{dest} already present, skipping')

In [ ]:
# Filter iNaturalist metadata to Florida ray-finned fish, then download the photos.
import os
import duckdb
from concurrent.futures import ThreadPoolExecutor

# ---- CONFIG ----
# Rough bounding box for Florida coastal / offshore waters.
FLORIDA_BBOX = {'min_lat': 24.3, 'max_lat': 31.1, 'min_lon': -87.7, 'max_lon': -79.7}

# Per-species caps. Drop species with too few photos to learn from, and cap the
# common ones so the dataset stays balanced (and the download stays reasonable).
MIN_PER_SPECIES = 50    # skip species with fewer than this many matching photos
MAX_PER_SPECIES = 300   # download at most this many photos per species

DATA_DIR = INAT_DATA_DIR                        # from the cell above
OUTPUT_DIR = '/content/Thalamus/data/raw_florida_fish'
os.makedirs(OUTPUT_DIR, exist_ok=True)

con = duckdb.connect(database=':memory:')

# DuckDB reads gzipped CSVs directly, no decompression needed.
for view in ['taxa', 'observations', 'photos']:
    con.execute(f"""
        CREATE VIEW {view} AS
        SELECT * FROM read_csv_auto('{DATA_DIR}/{view}.csv.gz', delim='\t', header=True)
    """)

# In this dataset `taxa.ancestry` is a slash-delimited string of taxon IDs
# (e.g. '48460/1/2/355675'), NOT names. So we first look up the taxon_id for the
# ray-finned fish class, then match descendants whose ancestry contains that id.
fish_taxon_id = con.execute(
    "SELECT taxon_id FROM taxa WHERE name = 'Actinopterygii'"
).fetchone()
if fish_taxon_id is None:
    raise SystemExit("Could not find the 'Actinopterygii' taxon — check taxa.csv.gz")
fish_taxon_id = fish_taxon_id[0]
print(f'Actinopterygii taxon_id = {fish_taxon_id}')

# Fish class (Actinopterygii, by id) + Florida bounding box + research-grade only.
# Then apply per-species caps with window functions:
#   - species_total  : how many photos the species has (for the MIN filter)
#   - rn             : per-species row number (for the MAX cap)
query = f"""
    WITH matched AS (
        SELECT
            o.observation_uuid,
            t.name AS species_name,
            o.latitude,
            o.longitude,
            p.photo_id,
            p.extension
        FROM observations o
        JOIN taxa t ON o.taxon_id = t.taxon_id
        JOIN photos p ON o.observation_uuid = p.observation_uuid
        WHERE ('/' || t.ancestry || '/') LIKE '%/{fish_taxon_id}/%'
          AND o.latitude BETWEEN {FLORIDA_BBOX['min_lat']} AND {FLORIDA_BBOX['max_lat']}
          AND o.longitude BETWEEN {FLORIDA_BBOX['min_lon']} AND {FLORIDA_BBOX['max_lon']}
          AND o.quality_grade = 'research'
    ),
    ranked AS (
        SELECT
            *,
            COUNT(*)     OVER (PARTITION BY species_name) AS species_total,
            ROW_NUMBER() OVER (PARTITION BY species_name ORDER BY photo_id) AS rn
        FROM matched
    )
    SELECT observation_uuid, species_name, latitude, longitude, photo_id, extension
    FROM ranked
    WHERE species_total >= {MIN_PER_SPECIES}
      AND rn <= {MAX_PER_SPECIES}
"""

results = con.execute(query).fetchdf()
print(
    f"Selected {len(results)} photos across {results['species_name'].nunique()} species "
    f"(min {MIN_PER_SPECIES}/species, capped at {MAX_PER_SPECIES})"
)
results.to_csv('/content/Thalamus/florida_fish_photo_list.csv', index=False)

# ---- DOWNLOAD MATCHING PHOTOS (via boto3, unsigned, in parallel) ----
import boto3
from botocore import UNSIGNED
from botocore.config import Config

MAX_WORKERS = 16
BUCKET = 'inaturalist-open-data'

# Size the HTTP connection pool to the worker count, otherwise boto3's default
# pool of 10 is too small for our threads and logs "Connection pool is full,
# discarding connection" while it churns connections instead of reusing them.
s3 = boto3.client('s3', config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=MAX_WORKERS,
))

def download_photo(row, size='medium'):
    species_dir = os.path.join(OUTPUT_DIR, row['species_name'].replace(' ', '_'))
    os.makedirs(species_dir, exist_ok=True)
    dest = os.path.join(species_dir, f"{row['photo_id']}.{row['extension']}")
    if os.path.exists(dest):
        return
    key = f"photos/{row['photo_id']}/{size}.{row['extension']}"
    try:
        s3.download_file(BUCKET, key, dest)
    except Exception as e:
        print(f"  skip {row['photo_id']}: {e}")

rows = results.to_dict('records')
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for i, _ in enumerate(pool.map(download_photo, rows), 1):
        if i % 5000 == 0:
            print(f'  {i}/{len(rows)} processed')

print(f'Done. Images saved under {OUTPUT_DIR}, organized by species folder.')

## 2. Clone the repo and install dependencies

Colab ships with torch already installed, so `torch` is skipped here to avoid a
long reinstall that usually resolves to the same version.

In [ ]:
import os

REPO_URL = 'https://github.com/conradpower279/Thalamus.git'
REPO_DIR = '/content/Thalamus'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull

In [ ]:
# timm/albumentations/onnx pinned to match requirements.txt; torch comes from Colab.
!pip install -q timm==1.0.11 albumentations==1.4.21 onnx==1.17.0 PyYAML==6.0.2

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Get the data

Colab's local disk is wiped when the runtime recycles, so keep the dataset in
Drive and copy it in. The archive should unpack to the layout described in
`data/README.md`:

```
data/train/<species>/*.jpg
data/val/<species>/*.jpg
```

Copying a zip and unpacking to local disk is much faster than reading thousands
of small files directly off a mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust to wherever your dataset archive lives in Drive.
ARCHIVE = '/content/drive/MyDrive/thalamus/fish_dataset.zip'

!mkdir -p /content/Thalamus/data
!unzip -q -o {ARCHIVE} -d /content/Thalamus/data
!ls /content/Thalamus/data

### Optional: build the splits

Only if your archive is one folder per species with no train/val/test split yet.

In [ ]:
# !python scripts/split_data.py --source /content/Thalamus/raw --output /content/Thalamus/data

### Check class balance

Worth a glance before every run. The training script uses unweighted
cross-entropy, so a heavily skewed dataset produces a model that quietly ignores
the rare species while still reporting good accuracy.

In [ ]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    split_dir = Path('data') / split
    if not split_dir.is_dir():
        continue
    counts = {
        d.name: len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png')))
        for d in sorted(split_dir.iterdir()) if d.is_dir()
    }
    print(f'{split}: {sum(counts.values())} images across {len(counts)} species')
    for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f'   {count:>6}  {name}')
    print()

## 4. Configure

Defaults come from `configs/default.yaml`. Review them here, then either edit
that file or pass overrides on the command line in the next cell.

In [ ]:
import yaml

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

for key, value in config.items():
    print(f'{key:>20}: {value}')

## 5. Train

Stage 1 trains the classifier head on the frozen backbone; stage 2 unfreezes
everything and fine-tunes with a cosine schedule. Checkpoints and the ONNX
export are written to `models/`.

Colab disconnects idle sessions — for long runs, keep the tab visible.

In [ ]:
!python scripts/fish_classifier_train.py --config configs/default.yaml --data-dir data

## 6. Back up the checkpoints

Do this before the runtime recycles, or the run is lost.

In [ ]:
!mkdir -p /content/drive/MyDrive/thalamus/models
!cp -v models/*.pth models/*.onnx models/classes.json /content/drive/MyDrive/thalamus/models/

## 7. Export to TFLite for the Flutter app

The conversion toolchain is large and conflicts with the training environment's
pins, so it is installed separately. Expect to restart the runtime afterwards.

For an int8 model, pass `--calib-dir data/val` — calibration needs real images.

In [ ]:
!pip install -q -r requirements-export.txt

In [ ]:
!python export/export_tflite.py --quantization float16
!ls -lh export/output/

## 8. Download the app artifacts

`classes.json` is not optional — the TFLite graph emits bare indices, and
without the mapping the app cannot name a species. Always keep the two together.

In [ ]:
from google.colab import files
from pathlib import Path

for path in sorted(Path('export/output').glob('*.tflite')):
    files.download(str(path))

files.download('export/output/classes.json')